# Araseの粒子データのplot、解析

In [ ]:
import os
import sys
from pathlib import Path

os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# geopack 1.0.13 と geopack-vectorize は同じ geopack 名前空間を使う。
# この notebook は trace_vectorized 等を使うため、workspace 内の
# geopack-vectorize 2.x を pyspedas より先に import できるようにする。
_gp_roots = [
    base / "geopack-vectorize"
    for base in (Path.cwd(), *Path.cwd().parents)
    if (base / "geopack-vectorize" / "geopack" / "__init__.py").is_file()
]
if not _gp_roots:
    raise FileNotFoundError("workspace 内に geopack-vectorize/geopack が見つからない")

_gp_root = _gp_roots[0].resolve()
_loaded_gp = sys.modules.get("geopack")
if _loaded_gp is not None:
    _loaded_path = Path(_loaded_gp.__file__).resolve()
    if _gp_root not in _loaded_path.parents:
        raise RuntimeError(
            "旧 geopack が既に import 済み。Kernel を再起動して先頭セルから実行する"
        )

if str(_gp_root) not in sys.path:
    sys.path.insert(0, str(_gp_root))

# LEP-e

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import numpy as np

time_range = ['20220901/21:00:00', '20220902/00:00:00']

ergpy.lepe(trange=time_range, datatype='3dflux', level='l2')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
import xarray as xr
import numpy as np

ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True)

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.erg_mgf_spintone_rm as emsr
import importlib
importlib.reload(emsr)

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

background_time_sec = 100 #[sec]

time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

In [ ]:
psp.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})
ergpy.orb(trange=time_range, level='l2', datatype='def')

In [ ]:
energy_list = psp.get_data('erg_lepe_l2_3dflux_FEDU', xarray=True).v1[0, :].data

energy_list = np.unique(np.sort(energy_list))

print(energy_list)

In [ ]:
#for i, energy in enumerate(energy_list):
#    psp.projects.erg.erg_lep_part_products(
#        'erg_lepe_l2_3dflux_FEDU',
#        outputs=['pa'],
#        energy=[np.trunc(energy), np.ceil(energy)],
#        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
#        pos_name='erg_orb_l2_pos_gse',
#        suffix='_'+str(i)
#    )

In [ ]:
#import numpy as np
#import xarray as xr
#import pandas as pd
#
#energy = np.asarray(energy_list, dtype=float)
#
#flux_list = []
#pa_ref = None
#
#for i, ene in enumerate(energy):
#    varname = f"erg_lepe_l2_3dflux_FEDU_pa_{i}"
#    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))
#
#    if da is None:
#        raise ValueError(f"{varname} が取得できない")
#
#    # flux: (time, v_dim) -> (time, energy, v_dim)
#    da_flux = da.expand_dims(energy=[ene])
#
#    flux_list.append(da_flux)
#
#    # pitch angle bin を確認
#    pa_now = da["spec_bins"].values   # shape=(time, v_dim)
#
#    if pa_ref is None:
#        pa_ref = pa_now
#    else:
#        if not np.allclose(pa_now, pa_ref, equal_nan=True):
#            raise ValueError(
#                f"{varname} の spec_bins が他の energy channel と一致しない"
#            )
#
## concat 後、(energy, time, v_dim) なので並べ替え
#da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")
#
## pitch angle が time に依らず一定か確認
#if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
#    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")
#
#pitch_angle = pa_ref[0, :]   # shape=(v_dim,)
#
## DataArray 化
#da_flux_3d = xr.DataArray(
#    da_flux_all.values,
#    dims=("time", "energy", "pitch_angle"),
#    coords={
#        "time": da_flux_all["time"].values,
#        "energy": energy,
#        "pitch_angle": pitch_angle,
#    },
#    name="FEDU_flux",
#    attrs={
#        "units": "#/s/cm^2/sr/eV",
#        "description": "LEP-e 3dflux as a function of time, energy, and pitch angle",
#    }
#)
#
#print(da_flux_3d)
#
#from pathlib import Path
#import pandas as pd
#
#t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
#t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")
#
#save_path = Path(
#    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPe_da_pa_energy_{t0_str}_{t1_str}.nc"
#)
#
#da_flux_3d.to_netcdf(save_path)
#print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPe_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPe_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPe = xr.open_dataset(LEPe_flux_Path)

print(da_LEPe)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import pandas as pd

#def plot_lepe_pitchangle_polar(
#    da_LEPe,
#    time,
#    ylabel,
#    vmin=1e2,
#    vmax=1e5,
#    emin=1e1,
#    emax=1e4,
#    cmap="turbo",
#    figsize=(6, 6),
#):
#    time = pd.Timestamp(time)
#
#    da_plot = (
#        da_LEPe["FEDU_flux"]
#        .sel(time=time, method="nearest")
#        .transpose("energy", "pitch_angle")
#    )
#
#    time_nearest = pd.Timestamp(da_plot.time.values).round("s")
#    time_nearest_next = time_nearest + pd.Timedelta(seconds=8)
#
#    E = da_plot["energy"].values
#    alpha_deg = da_plot["pitch_angle"].values
#    flux = da_plot.values.astype(float)
#
#    flux[~np.isfinite(flux)] = np.nan
#    flux[flux <= 0] = np.nan
#
#    def make_edges(x):
#        x = np.asarray(x, dtype=float)
#        dx = np.diff(x)
#        x_edge = np.empty(x.size + 1, dtype=float)
#        x_edge[1:-1] = 0.5 * (x[:-1] + x[1:])
#        x_edge[0] = x[0] - 0.5 * dx[0]
#        x_edge[-1] = x[-1] + 0.5 * dx[-1]
#        return x_edge
#
#    E_edge = make_edges(E)
#    alpha_edge = np.deg2rad(make_edges(alpha_deg))
#
#    TT, RR = np.meshgrid(alpha_edge, E_edge, indexing="xy")
#
#    mpl.rcParams["font.size"] = 20
#
#    fig = plt.figure(figsize=figsize)
#    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.03], wspace=0)
#
#    ax_0 = fig.add_subplot(gs[0, 0], projection="polar")
#    cax_0 = fig.add_subplot(gs[0, 1])
#
#    ax_0.set_theta_zero_location("N")
#    ax_0.set_theta_direction(-1)
#    ax_0.set_thetamin(0)
#    ax_0.set_thetamax(180)
#
#    ax_0.set_rscale("log")
#    ax_0.set_rlabel_position(185)
#    ax_0.set_ylim(emin, emax)
#    ax_0.set_ylabel(ylabel, labelpad=-30)
#
#    mesh = ax_0.pcolormesh(
#        TT,
#        RR,
#        flux,
#        cmap=cmap,
#        norm=mcolors.LogNorm(vmin=vmin, vmax=vmax),
#        shading="flat",
#    )
#
#    title_str = f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}"
#    ax_0.set_title(title_str, pad=20)
#
#    cb = fig.colorbar(mesh, cax=cax_0)
#    cb.set_label(r'[$\mathrm{s}^{-1} \mathrm{cm}^{-2} \mathrm{str}^{-1} \mathrm{eV}^{-1}$]')
#
#    ax_0.minorticks_on()
#    ax_0.set_thetagrids(np.rad2deg(np.linspace(0, np.pi, 7)))
#    ax_0.grid(True, which="both", linestyle=":", alpha=0.5)
#    ax_0.grid(True, which="major", linestyle="solid", alpha=1, c="k")
#    ax_0.tick_params(axis="x", pad=10)
#
#    fig.subplots_adjust(left=0.11, right=0.87)
#
#    return fig, ax_0, cax_0

# loss cone angleを求める

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import importlib
import sys
from pathlib import Path

# 旧 geopack 1.0.13 が先に読み込まれていても、このセルだけで復旧する。
_gp_roots = [
    base / "geopack-vectorize"
    for base in (Path.cwd(), *Path.cwd().parents)
    if (base / "geopack-vectorize" / "geopack" / "__init__.py").is_file()
]
if not _gp_roots:
    raise FileNotFoundError("workspace 内に geopack-vectorize/geopack が見つからない")
_gp_root = _gp_roots[0].resolve()

# PathFinder が site-packages の旧版を拾わないよう、ローカル版を常に先頭へ置く。
sys.path[:] = [p for p in sys.path if p != str(_gp_root)]
sys.path.insert(0, str(_gp_root))

_loaded_gp = sys.modules.get("geopack")
if _loaded_gp is not None:
    _loaded_path = Path(_loaded_gp.__file__).resolve()
    if _gp_root not in _loaded_path.parents:
        for _name in tuple(sys.modules):
            if _name == "geopack" or _name.startswith("geopack."):
                del sys.modules[_name]

importlib.invalidate_caches()
geopack = importlib.import_module("geopack")
_required_geopack_api = {"trace_vectorized", "smgsm_vectorized", "t04", "igrf_gsm"}
_missing_geopack_api = sorted(_required_geopack_api - set(dir(geopack)))
if _missing_geopack_api:
    raise ImportError(
        f"geopack-vectorize 2.x が読み込まれていない: missing={_missing_geopack_api}. "
        "Kernel を再起動し、先頭セルから実行する"
    )
from geopack import trace_vectorized, smgsm_vectorized
from geopack import t04, igrf_gsm
from pyspedas.geopack.get_tsy_params import get_tsy_params


earth_radius = 6378.1  # km


def datetime64_to_unix_seconds(t):
    return pd.Timestamp(t).timestamp()


def _as_time_index(times):
    """
    scalar time, list-like time, DatetimeIndex をすべて DatetimeIndex にする。
    """
    if isinstance(times, pd.DatetimeIndex):
        return times

    t = pd.to_datetime(times)

    if np.isscalar(t) or isinstance(t, pd.Timestamp):
        return pd.DatetimeIndex([t])

    return pd.DatetimeIndex(t)


def _interp_to_times(da, trace_times):
    trace_times = _as_time_index(trace_times)
    return da.interp(time=trace_times)


def _to_2d_parmod(ts04_par, trace_times):
    trace_times = _as_time_index(trace_times)

    if isinstance(ts04_par, xr.DataArray):
        if "time" in ts04_par.coords:
            par = ts04_par.interp(time=trace_times).values
        else:
            par = ts04_par.values
    else:
        par = np.asarray(ts04_par)

    par = np.asarray(par, dtype=float)

    if par.ndim == 1:
        if par.size != 10:
            raise ValueError("T04 parmod must have length 10.")
        par = np.tile(par, (len(trace_times), 1))

    if par.shape != (len(trace_times), 10):
        raise ValueError(f"parmod shape must be ({len(trace_times)}, 10), got {par.shape}")

    return par


def _interp_Bsc_to_times(B_sc, trace_times):
    """
    B_sc: xarray.DataArray(time) or array-like or scalar
          unit: nT
    """
    trace_times = _as_time_index(trace_times)

    if isinstance(B_sc, xr.DataArray):
        B_i = B_sc.interp(time=trace_times).values
    else:
        B_i = np.asarray(B_sc, dtype=float)

    if np.ndim(B_i) == 0:
        B_i = np.full(len(trace_times), float(B_i))

    B_i = np.asarray(B_i, dtype=float)

    if B_i.shape != (len(trace_times),):
        raise ValueError(f"B_sc shape must be ({len(trace_times)},), got {B_i.shape}")

    return B_i


def model_B_t04_total_gsm(x, y, z, parmod):
    """
    GSM, R_E の位置で T04 external + IGRF internal の磁場を計算する。
    戻り値: Bx, By, Bz, |B| [nT]

    注意:
    geopack の実装では t04 は外部磁場、igrf_gsm は内部磁場。
    """
    bx_ext, by_ext, bz_ext = t04(parmod, 0.0, x, y, z)
    bx_int, by_int, bz_int = igrf_gsm(x, y, z)

    bx = bx_ext + bx_int
    by = by_ext + by_int
    bz = bz_ext + bz_int

    babs = np.sqrt(bx**2 + by**2 + bz**2)

    return bx, by, bz, babs


def trace_to_loss_altitude_t04(
    pos_gsm,
    ts04_par,
    B_sc,
    trace_times=None,
    loss_alt_km=500.0,
    directions=(+1, -1),
    exname="t04",
    inname="igrf",
    rlim=30.0,
    maxloop=5000,
    strict_scalar_models=False,
    verbose=True,
):
    """
    衛星位置から T04+IGRF で高度 loss_alt_km まで field line trace し、
    footprint 磁場 B_fp と loss cone angle を計算する。

    Parameters
    ----------
    pos_gsm : xarray.DataArray
        衛星位置。shape=(time, 3), GSM, R_E。
    ts04_par : xarray.DataArray or ndarray
        T04 parmod。shape=(time, 10) or (10,)。
        [Pdyn, Dst, ByIMF, BzIMF, W1, W2, W3, W4, W5, W6]
    B_sc : xarray.DataArray or ndarray or scalar
        衛星位置での観測磁場強度 [nT]。
        すでに得られている衛星磁場データの |B| を渡す。
    trace_times : array-like or None
        計算時刻。Noneなら pos_gsm.time を使う。
    loss_alt_km : float
        ロス境界高度 [km]。今回は 500 km。
    directions : tuple
        (+1, -1) なら両半球へ trace。
    """

    if trace_times is None:
        trace_times = _as_time_index(pos_gsm["time"].values)
    else:
        trace_times = _as_time_index(trace_times)

    pos_i = _interp_to_times(pos_gsm, trace_times)
    par_i = _to_2d_parmod(ts04_par, trace_times)
    B_sc_i = _interp_Bsc_to_times(B_sc, trace_times)

    r0_loss = (earth_radius + loss_alt_km) / earth_radius

    results = []

    for it, t in enumerate(trace_times):
        x0, y0, z0 = np.asarray(pos_i.values[it, :], dtype=float)
        parmod = np.asarray(par_i[it, :], dtype=float)
        bsc = float(B_sc_i[it])

        if (
            not np.all(np.isfinite([x0, y0, z0]))
            or not np.all(np.isfinite(parmod))
            or not np.isfinite(bsc)
            or bsc <= 0
        ):
            if verbose:
                print(f"skip {t}: bad input")
                print("pos_gsm =", [x0, y0, z0])
                print("parmod  =", parmod)
                print("B_sc    =", bsc)
            continue

        geopack.recalc(datetime64_to_unix_seconds(t))

        for direction in directions:
            try:
                ret = trace_vectorized(
                    x0, y0, z0,
                    dir=direction,
                    rlim=rlim,
                    r0=r0_loss,
                    parmod=parmod,
                    exname=exname,
                    inname=inname,
                    maxloop=maxloop,
                    return_full_path=True,
                    strict_scalar_models=strict_scalar_models
                )

                # 君の環境に合わせて:
                # xf, yf, zf, xx, yy, zz, status
                xf, yf, zf, xx, yy, zz, status = ret[:7]

                xf = float(np.asarray(xf).squeeze())
                yf = float(np.asarray(yf).squeeze())
                zf = float(np.asarray(zf).squeeze())
                status = int(np.asarray(status).squeeze())

                # footprintでのT04+IGRF磁場
                geopack.recalc(datetime64_to_unix_seconds(t))
                bx_fp, by_fp, bz_fp, B_fp = model_B_t04_total_gsm(
                    xf, yf, zf, parmod
                )

                ratio = bsc / B_fp

                if ratio < 0:
                    alpha_lc_deg = np.nan
                elif ratio > 1:
                    # 衛星位置のBがfootpointより大きい場合。
                    # 通常のloss coneとしては物理的に怪しいので90度に丸めずNaNにする。
                    alpha_lc_deg = np.nan
                else:
                    alpha_lc_deg = np.rad2deg(np.arcsin(np.sqrt(ratio)))

                # footprint GSM -> SM
                xfp_sm, yfp_sm, zfp_sm = smgsm_vectorized(xf, yf, zf, j=-1)

                results.append({
                    "time": t,
                    "direction": direction,
                    "status": status,

                    "x_sc_gsm": x0,
                    "y_sc_gsm": y0,
                    "z_sc_gsm": z0,

                    "x_fp_gsm": xf,
                    "y_fp_gsm": yf,
                    "z_fp_gsm": zf,

                    "x_fp_sm": float(np.asarray(xfp_sm)),
                    "y_fp_sm": float(np.asarray(yfp_sm)),
                    "z_fp_sm": float(np.asarray(zfp_sm)),

                    "r_fp_re": np.sqrt(xf**2 + yf**2 + zf**2),
                    "alt_fp_km": (np.sqrt(xf**2 + yf**2 + zf**2) - 1.0) * earth_radius,

                    "Bx_fp_nT": bx_fp,
                    "By_fp_nT": by_fp,
                    "Bz_fp_nT": bz_fp,
                    "B_fp_nT": B_fp,

                    "B_sc_nT": bsc,
                    "B_sc_over_B_fp": ratio,
                    "alpha_lc_deg": alpha_lc_deg,

                    "parmod": parmod
                })

                if verbose:
                    print(
                        f"{t}, dir={direction}, status={status}, "
                        f"B_sc={bsc:.2f} nT, B_fp={B_fp:.2f} nT, "
                        f"alpha_LC={alpha_lc_deg:.2f} deg, "
                        f"alt_fp={results[-1]['alt_fp_km']:.1f} km"
                    )

            except Exception as e:
                if verbose:
                    print(f"failed {t}, dir={direction}: {e}")

    return results

def split_losscone_by_hemisphere(losscone_data):
    """
    trace_to_loss_altitude_t04 の結果を北半球・南半球に分ける。

    判定:
        footprint の z_fp_sm > 0 なら north
        footprint の z_fp_sm < 0 なら south

    Returns
    -------
    out : dict
        {
            "north": dict or None,
            "south": dict or None,
            "alpha_lc_north_deg": float,
            "alpha_lc_south_deg": float,
        }
    """
    north = None
    south = None

    for d in losscone_data:
        zfp = d.get("z_fp_sm", np.nan)

        if not np.isfinite(zfp):
            continue

        if zfp > 0:
            north = d
        elif zfp < 0:
            south = d

    out = {
        "north": north,
        "south": south,
        "alpha_lc_north_deg": np.nan if north is None else north["alpha_lc_deg"],
        "alpha_lc_south_deg": np.nan if south is None else south["alpha_lc_deg"],
    }

    return out

In [ ]:
trange_param = ['2022-09-01/00:00', '2022-09-02/00:00']

psp.projects.kyoto.dst(trange=trange_param)
psp.projects.omni.data(trange=trange_param)

psp.join_vec(['BX_GSE', 'BY_GSM', 'BZ_GSM'])

params_name = get_tsy_params(
    dst_tvar='kyoto_dst',
    imf_tvar='BX_GSE-BY_GSM-BZ_GSM_joined',
    Np_tvar='proton_density',
    Vp_tvar='flow_speed',
    model='ts04',
    pressure_tvar='Pressure',
    speed=True,
    newname='ts04_par'
)

ts04_par = psp.get_data(params_name, xarray=True)
print(ts04_par)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import pandas as pd

erg_pos_gsm = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)

def plot_lepe_pitchangle_polar(
    da_LEPe,
    time,
    ylabel,
    vmin=1e2,
    vmax=1e5,
    emin=1e1,
    emax=1e4,
    cmap="turbo",
    figsize=(6, 6),
    pa_range_for_stats=(10, 170),   # 統計に使うpitch angle範囲
    overlay_peak_line=True,
    mass=9.1093837E-31,
    VDF_convert=False,
    loss_cone_plot=False,
):
    time = pd.Timestamp(time)

    da_plot = (
        da_LEPe["FEDU_flux"]
        .sel(time=time, method="nearest")
        .transpose("energy", "pitch_angle")
    )

    time_nearest = pd.Timestamp(da_plot.time.values).round("s")
    time_nearest_next = time_nearest + pd.Timedelta(seconds=8)

    E = da_plot["energy"].values                  # shape: (nE,)
    alpha_deg = da_plot["pitch_angle"].values     # shape: (nPA,)
    flux = da_plot.values.astype(float)           # shape: (nE, nPA)

    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan

    if VDF_convert==True:
        # differential number flux convert to velocity distribution function
        flux = 5E3 / E[:, None] * (mass / 1.60218E-19)**2. * flux  # [s3 m-6]

    def make_edges(x):
        x = np.asarray(x, dtype=float)
        if x.size < 2:
            raise ValueError("Need at least 2 points to make edges.")
        dx = np.diff(x)
        x_edge = np.empty(x.size + 1, dtype=float)
        x_edge[1:-1] = 0.5 * (x[:-1] + x[1:])
        x_edge[0] = x[0] - 0.5 * dx[0]
        x_edge[-1] = x[-1] + 0.5 * dx[-1]
        return x_edge

    E_edge = make_edges(E)
    alpha_edge = np.deg2rad(make_edges(alpha_deg))
    TT, RR = np.meshgrid(alpha_edge, E_edge, indexing="xy")

    mpl.rcParams["font.size"] = 20

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.03], wspace=0)

    ax_0 = fig.add_subplot(gs[0, 0], projection="polar")
    cax_0 = fig.add_subplot(gs[0, 1])

    ax_0.set_theta_zero_location("N")
    ax_0.set_theta_direction(-1)
    ax_0.set_thetamin(0)
    ax_0.set_thetamax(180)

    ax_0.set_rscale("log")
    ax_0.set_rlabel_position(185)
    ax_0.set_ylim(emin, emax)
    ax_0.set_ylabel(ylabel, labelpad=-30)

    mesh = ax_0.pcolormesh(
        TT,
        RR,
        flux,
        cmap=cmap,
        norm=mcolors.LogNorm(vmin=vmin, vmax=vmax),
        shading="flat",
    )

    if VDF_convert==False:
        # -----------------------------
        # 各 pitch angle におけるピーク energy を求める
        # -----------------------------
        peak_energy = np.full(alpha_deg.shape, np.nan, dtype=float)
        peak_flux = np.full(alpha_deg.shape, np.nan, dtype=float)

        for j in range(len(alpha_deg)):
            col = flux[:, j]
            valid = np.isfinite(col) & np.isfinite(E) & (E >= emin) & (E <= emax)
            if np.any(valid):
                idx_local = np.nanargmax(col[valid])
                E_valid = E[valid]
                F_valid = col[valid]
                peak_energy[j] = E_valid[idx_local]
                peak_flux[j] = F_valid[idx_local]

        # -----------------------------
        # 線を重ねる
        # -----------------------------
        if overlay_peak_line:
            valid_line = np.isfinite(peak_energy)
            ax_0.plot(
                np.deg2rad(alpha_deg[valid_line]),
                peak_energy[valid_line],
                color="k",
                lw=2.0,
                marker="o",
                ms=3,
                zorder=5,
            )

        # -----------------------------
        # 代表値を算出
        # -----------------------------
        pa_min, pa_max = pa_range_for_stats
        valid_stats = (
            np.isfinite(peak_energy)
            & np.isfinite(peak_flux)
            & (alpha_deg >= pa_min)
            & (alpha_deg <= pa_max)
        )

        # 単純平均より、ピークflux重み付き平均の方が ring の主成分を反映しやすい
        if np.any(valid_stats):
            weights = peak_flux[valid_stats].copy()
            weights = np.where(np.isfinite(weights) & (weights > 0), weights, 0.0)

            if np.sum(weights) > 0:
                E_mean = np.sum(weights * peak_energy[valid_stats]) / np.sum(weights)
                E_std = np.sqrt(
                    np.sum(weights * (peak_energy[valid_stats] - E_mean) ** 2) / np.sum(weights)
                )
            else:
                E_mean = np.nanmean(peak_energy[valid_stats])
                E_std = np.nanstd(peak_energy[valid_stats])
        else:
            E_mean = np.nan
            E_std = np.nan
    
    elif VDF_convert==True:
        E_mean=np.nan

    if loss_cone_plot == True:
        da_B_background_abs = np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
        losscone_data = trace_to_loss_altitude_t04(
            pos_gsm     = erg_pos_gsm,
            ts04_par    = ts04_par,
            B_sc        = da_B_background_abs,
            trace_times = time,
            loss_alt_km = 500.0,
            directions  = (+1, -1),
            verbose     = True
        )
        losscone_hemi = split_losscone_by_hemisphere(losscone_data)

        alpha_lc_north = losscone_hemi["alpha_lc_north_deg"]
        alpha_lc_south = 180. - losscone_hemi["alpha_lc_south_deg"]

        ax_0.plot([np.deg2rad(alpha_lc_north), np.deg2rad(alpha_lc_north)], [1E1, 1E4], color='magenta', linestyle='dashed', lw=2)
        ax_0.plot([np.deg2rad(alpha_lc_south), np.deg2rad(alpha_lc_south)], [1E1, 1E4], color='magenta', linestyle='dashed', lw=2)

    elif loss_cone_plot==False:
        alpha_lc_north = np.nan
        alpha_lc_south = np.nan



    # -----------------------------
    # title
    # -----------------------------
    title_str = (
        f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}\n"
        f"{E_mean:.2f} ± {E_std:.2f} eV"
        if np.isfinite(E_mean)
        else f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}"
    )
    
    if np.isfinite(alpha_lc_north):
        title_str += f"\n loss cone: {alpha_lc_north:.2f}° / {alpha_lc_south:.2f}°"
    ax_0.set_title(title_str, pad=20)

    cb = fig.colorbar(mesh, cax=cax_0)
    if VDF_convert==False:
        cb.set_label(r'[$\mathrm{s}^{-1} \mathrm{cm}^{-2} \mathrm{str}^{-1} \mathrm{eV}^{-1}$]')
    elif VDF_convert==True:
        cb.set_label(r'[$\mathrm{s}^{3} \mathrm{m}^{-6}$]')

    ax_0.minorticks_on()
    ax_0.set_thetagrids(np.rad2deg(np.linspace(0, np.pi, 7)))
    ax_0.grid(True, which="both", linestyle=":", alpha=0.5)
    ax_0.grid(True, which="major", linestyle="solid", alpha=1, c="k")
    ax_0.tick_params(axis="x", pad=10)

    fig.subplots_adjust(left=0.11, right=0.87)

    if VDF_convert==False:
        results = {
            "time_nearest": time_nearest,
            "time_nearest_next": time_nearest_next,
            "pitch_angle_deg": alpha_deg,
            "peak_energy_eV": peak_energy,
            "peak_flux": peak_flux,
            "E_ring_mean_eV": E_mean,
            "E_ring_std_eV": E_std,
            "VDF_convert": VDF_convert,
        }
    elif VDF_convert==True:
        results = {
            "time_nearest": time_nearest,
            "time_nearest_next": time_nearest_next,
            "pitch_angle_deg": alpha_deg,
            "VDF_convert": VDF_convert,
        }

    return fig, ax_0, cax_0, results

In [ ]:
fig, ax, cax, results = plot_lepe_pitchangle_polar(
    da_LEPe=da_LEPe,
    time="2022-09-01T22:37:00",
    ylabel=r"$\mathrm{e}^{-}$ Energy [eV]",
    pa_range_for_stats=(10, 170),
    vmin=1e-21,
    vmax=1e-15,
    mass=9.1093837E-31,
    VDF_convert=True,
    loss_cone_plot=True
)
plt.show()
#print("Representative ring energy:")
#print(f"{results['E_ring_mean_eV']:.1f} ± {results['E_ring_std_eV']:.1f} eV")
print(results)


In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#ylabel_lepe = r"$\mathrm{e}^{-}$ Energy [eV]"
#
#out_dir = f'/mnt/j/KAW_observation/LEP-e_pitch_angle_each_time/20220901/21-24_energy_pa_VDF_lc'
#os.makedirs(out_dir, exist_ok=True)
#
#out_dir_pdf = f'{out_dir}/PDF/'
#os.makedirs(out_dir_pdf, exist_ok=True)
#out_dir_png = f'{out_dir}/PNG/'
#os.makedirs(out_dir_png, exist_ok=True)
#
#csv_path = f'{out_dir}/ring_energy_summary.csv'
#
## 毎回全配列に percentiles を掛けると無駄なので先に一度だけ計算
#flux_all    = np.where(da_LEPe.FEDU_flux.values > 0, da_LEPe.FEDU_flux.values, np.nan)
#VDF_all     = flux_all * 5E3 / da_LEPe["energy"].values[:, None] * (9.1093837E-31 / 1.60218E-19)**2.  # [s3 m-6]
#vmin_global = np.nanpercentile(VDF_all, 50)
#vmax_global = np.nanpercentile(VDF_all, 99)
#
#def process_and_save_plot(t):
#    ts = pd.Timestamp(t)
#
#    fig = None
#    try:
#        fig, ax, cax, results = plot_lepe_pitchangle_polar(
#            da_LEPe=da_LEPe,
#            time=ts,
#            ylabel=ylabel_lepe,
#            vmin=vmin_global,
#            vmax=vmax_global,
#            mass=9.1093837E-31,
#            VDF_convert=True,
#            loss_cone_plot=True,
#        )
#
#        base = ts.strftime('%Y-%m-%dT%H%M%S')
#        png_path = os.path.join(out_dir_png, base + '.png')
#        pdf_path = os.path.join(out_dir_pdf, base + '.pdf')
#
#        fig.savefig(png_path, dpi=200, bbox_inches='tight')
#        fig.savefig(pdf_path, bbox_inches='tight')
#
#        # CSV 用の1行
#        row = {
#            "time_input": ts.isoformat(),
#            "time_nearest": pd.Timestamp(results["time_nearest"]).isoformat(),
#            "time_nearest_next": pd.Timestamp(results["time_nearest_next"]).isoformat(),
#            "E_ring_mean_eV": results["E_ring_mean_eV"],
#            "E_ring_std_eV": results["E_ring_std_eV"],
#        }
#
#        # pitch-angle ごとの peak energy も保存したいなら文字列化して持たせる
#        pa = np.asarray(results["pitch_angle_deg"], dtype=float)
#        epeak = np.asarray(results["peak_energy_eV"], dtype=float)
#        pflux = np.asarray(results["peak_flux"], dtype=float)
#
#        row["pitch_angle_deg"] = ",".join(
#            "nan" if not np.isfinite(x) else f"{x:.1f}" for x in pa
#        )
#        row["peak_energy_eV_by_pa"] = ",".join(
#            "nan" if not np.isfinite(x) else f"{x:.3f}" for x in epeak
#        )
#        row["peak_flux_by_pa"] = ",".join(
#            "nan" if not np.isfinite(x) else f"{x:.6g}" for x in pflux
#        )
#
#        return row
#
#    except Exception as e:
#        # 並列処理ではどの時刻で落ちたか分かるようにして返す
#        return {
#            "time_input": ts.isoformat(),
#            "time_nearest": None,
#            "time_nearest_next": None,
#            "E_ring_mean_eV": np.nan,
#            "E_ring_std_eV": np.nan,
#            "pitch_angle_deg": None,
#            "peak_energy_eV_by_pa": None,
#            "peak_flux_by_pa": None,
#            "error": repr(e),
#        }
#
#    finally:
#        if fig is not None:
#            plt.close(fig)
#
#time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_grid = da_LEPe.sel(time=slice(t_min, t_max)).time.values
#
#with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame"):
#    results_list = Parallel(
#        n_jobs=os.cpu_count(),
#        backend="loky",
#        verbose=0
#    )(
#        delayed(process_and_save_plot)(t) for t in time_grid
#    )
#
#df_results = pd.DataFrame(results_list)
#df_results = df_results.sort_values("time_input").reset_index(drop=True)
#df_results.to_csv(csv_path, index=False)
#
#print(f"Finished saving all plots and CSV: {csv_path}")

## LEP-e L3 pitch-angle distribution

L3 `FEDU` を既存の `(time, energy, pitch_angle)` 形式へ整形し、同じ極座標図と loss cone 計算へ渡す。

In [ ]:
# LEP-e L3 PA: FEDU [#/s-cm2-sr-eV] -> xarray.Dataset
lepe_l3_raw = psp.projects.erg.lepe(
    trange=time_range,
    level="l3",
    datatype="pa",
    notplot=True,
    time_clip=True,
)

l3_name = "erg_lepe_l3_pa_FEDU"
if l3_name not in lepe_l3_raw:
    raise KeyError(f"{l3_name} was not loaded: {list(lepe_l3_raw)}")

l3 = lepe_l3_raw[l3_name]
flux_l3 = np.asarray(l3["y"], dtype=float)
energy_l3 = np.asarray(l3["v1"], dtype=float)
pa_l3 = np.asarray(l3["v2"], dtype=float)

if flux_l3.ndim != 3:
    raise ValueError(f"FEDU must be (time, energy, pitch_angle), got {flux_l3.shape}")
if energy_l3.ndim != 1 or pa_l3.ndim != 1:
    raise ValueError(f"Expected 1-D energy/PA coordinates, got {energy_l3.shape}, {pa_l3.shape}")
if flux_l3.shape[1:] != (energy_l3.size, pa_l3.size):
    raise ValueError("FEDU dimensions do not match the energy and PA coordinates")

cdf_meta_l3 = l3.get("CDF", {})
units_l3 = cdf_meta_l3.get("VATT", {}).get("UNITS", "")
units_key_l3 = units_l3.lower().replace(" ", "")
if "kev" in units_key_l3:
    # per keV -> per eV
    flux_l3 *= 1e-3
elif "ev" not in units_key_l3:
    raise ValueError(f"Unsupported/unknown FEDU energy unit: {units_l3!r}")

# pyspedas versions may return CDF epoch as datetime64 or Unix seconds.
epoch_l3 = np.asarray(l3["x"])
if np.issubdtype(epoch_l3.dtype, np.datetime64):
    time_l3 = pd.to_datetime(epoch_l3)
else:
    time_l3 = pd.to_datetime(epoch_l3.astype(float), unit="s")
# v05_01 has two unused energy slots represented by NaN; do not pass them to pcolormesh.
valid_e = np.isfinite(energy_l3) & (energy_l3 > 0)
order_e = np.flatnonzero(valid_e)[np.argsort(energy_l3[valid_e])]
order_pa = np.argsort(pa_l3)

da_LEPe_L3 = xr.Dataset(
    {
        "FEDU_flux": (
            ("time", "energy", "pitch_angle"),
            flux_l3[:, order_e, :][:, :, order_pa],
        )
    },
    coords={
        "time": time_l3,
        "energy": energy_l3[order_e],
        "pitch_angle": pa_l3[order_pa],
    },
    attrs={
        "source": l3_name,
        "source_units": units_l3,
        "data_version": cdf_meta_l3.get("GATT", {}).get("DATA_VERSION", [None])[0],
        "time_resolution": cdf_meta_l3.get("GATT", {}).get("TIME_RESOLUTION", [None])[0],
    },
).sel(time=slice(*pd.to_datetime(time_range)))

print(da_LEPe_L3)
print("source units:", units_l3)
print("version:", da_LEPe_L3.attrs["data_version"])
print("time resolution:", da_LEPe_L3.attrs["time_resolution"])

In [ ]:
# 単発確認: 既存と同じ図形式、VDF、TS04+IGRF loss cone
positive_l3 = da_LEPe_L3["FEDU_flux"].where(da_LEPe_L3["FEDU_flux"] > 0)
vdf_l3 = (
    positive_l3
    * 5e3 / da_LEPe_L3["energy"]
    * (9.1093837e-31 / 1.602176634e-19) ** 2
)
vmin_l3 = float(vdf_l3.quantile(0.50, skipna=True))
vmax_l3 = float(vdf_l3.quantile(0.99, skipna=True))

fig, ax, cax, results_l3 = plot_lepe_pitchangle_polar(
    da_LEPe=da_LEPe_L3,
    time="2022-09-01T22:37:00",
    ylabel=r"$\mathrm{e}^{-}$ Energy [eV]",
    vmin=vmin_l3,
    vmax=vmax_l3,
    mass=9.1093837e-31,
    VDF_convert=True,
    loss_cone_plot=True,
)
plt.show()
print(results_l3)

In [ ]:
# 全L3時刻を既存と同じPNG/PDF形式で保存
out_dir_l3 = "/mnt/j/KAW_observation/LEP-e_L3_pitch_angle_each_time/20220901/21-24_energy_pa_VDF_lc"
out_dir_l3_pdf = os.path.join(out_dir_l3, "PDF")
out_dir_l3_png = os.path.join(out_dir_l3, "PNG")
os.makedirs(out_dir_l3_pdf, exist_ok=True)
os.makedirs(out_dir_l3_png, exist_ok=True)

def process_and_save_plot_lepe_l3(t):
    ts = pd.Timestamp(t)
    fig = None
    try:
        fig, ax, cax, result = plot_lepe_pitchangle_polar(
            da_LEPe=da_LEPe_L3,
            time=ts,
            ylabel=r"$\mathrm{e}^{-}$ Energy [eV]",
            vmin=vmin_l3,
            vmax=vmax_l3,
            mass=9.1093837e-31,
            VDF_convert=True,
            loss_cone_plot=True,
        )
        base = ts.strftime("%Y-%m-%dT%H%M%S")
        fig.savefig(os.path.join(out_dir_l3_png, base + ".png"), dpi=200, bbox_inches="tight")
        fig.savefig(os.path.join(out_dir_l3_pdf, base + ".pdf"), bbox_inches="tight")
        return {"time_input": ts.isoformat(), "error": None}
    except Exception as exc:
        return {"time_input": ts.isoformat(), "error": repr(exc)}
    finally:
        if fig is not None:
            plt.close(fig)

time_grid_l3 = da_LEPe_L3.sel(time=slice(*pd.to_datetime(time_range))).time.values
with TqdmJoblib(total=len(time_grid_l3), desc="Saving LEP-e L3 plots", unit="frame"):
    results_l3_all = Parallel(n_jobs=os.cpu_count(), backend="loky", verbose=0)(
        delayed(process_and_save_plot_lepe_l3)(t) for t in time_grid_l3
    )

df_l3_status = pd.DataFrame(results_l3_all)
failed_l3 = df_l3_status[df_l3_status["error"].notna()]
print(f"Finished: {len(df_l3_status) - len(failed_l3)}/{len(df_l3_status)} frames")
display(failed_l3.head())

# LEP-i

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import numpy as np

ergpy.lepi(trange=time_range, datatype='3dflux', level='l2')

# $\mathrm{H}^{+}$

In [ ]:
energy_list_P = psp.get_data('erg_lepi_l2_3dflux_FPDU', xarray=True).v1 * 1E3

energy_list_P = np.unique(np.sort(energy_list_P))

print(energy_list_P)

In [ ]:
for i, energy in enumerate(energy_list_P):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FPDU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_P, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FPDU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FPDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FPDU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FPDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_P = xr.open_dataset(LEPi_FPDU_flux_Path)

print(da_LEPi_P)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

ylabel_lepi_P = r"$\mathrm{H}^{+}$ Energy [eV]"

out_dir = f'/mnt/j/KAW_observation/LEP-i_P_pitch_angle_each_time/20220901/21-24_energy_pa_VDF'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

flux_all    = np.where(da_LEPi_P.FEDU_flux.values > 0, da_LEPi_P.FEDU_flux.values, np.nan)
VDF_all     = flux_all * 5E3 / da_LEPi_P["energy"].values[:, None] * (1.6726219e-27 / 1.60218E-19)**2.  # [s3 m-6]
vmin_global = np.nanpercentile(VDF_all, 50)
vmax_global = np.nanpercentile(VDF_all, 99)
print(vmin_global, vmax_global)

def process_and_save_plot(t):

    fig, ax, cax, _ = plot_lepe_pitchangle_polar(
        da_LEPe=da_LEPi_P,
        time=t,
        ylabel=ylabel_lepi_P,
        vmin=vmin_global,
        vmax=vmax_global,
        mass=1.6726219e-27,
        VDF_convert=True,
    )
    if fig is None:
        return None

    ts = pd.Timestamp(t)
    base = ts.strftime('%Y-%m-%dT%H%M%S')
    png_path = os.path.join(out_dir_png, base + '.png')
    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

    try:
        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
    finally:
        plt.close(fig)

    return

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']

t_min, t_max    = pd.to_datetime(time_range)
time_grid = da_LEPi_P.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame"):
    results_list = Parallel(
        n_jobs=os.cpu_count(),
        backend="loky",
        verbose=0
    )(
        delayed(process_and_save_plot)(t) for t in time_grid
    )
print('Finished saving all plots!')

# $\mathrm{He}^{+}$

In [ ]:
energy_list_HE = psp.get_data('erg_lepi_l2_3dflux_FHEDU', xarray=True).v1 * 1E3

energy_list_HE = np.unique(np.sort(energy_list_HE))

print(energy_list_HE)

In [ ]:
for i, energy in enumerate(energy_list_HE):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FHEDU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_HE, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FHEDU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FHEDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FHEDU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FHEDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_HE = xr.open_dataset(LEPi_FHEDU_flux_Path)

print(da_LEPi_HE)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

ylabel_lepi_HE = r"$\mathrm{He}^{+}$ Energy [eV]"

out_dir = f'/mnt/j/KAW_observation/LEP-i_HE_pitch_angle_each_time/20220901/21-24_energy_pa_VDF'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

flux_all    = np.where(da_LEPi_HE.FEDU_flux.values > 0, da_LEPi_HE.FEDU_flux.values, np.nan)
VDF_all     = flux_all * 5E3 / da_LEPi_HE["energy"].values[:, None] * (1.6726219e-27 * 4. / 1.60218E-19)**2.  # [s3 m-6]
vmin_global = np.nanpercentile(VDF_all, 50)
vmax_global = np.nanpercentile(VDF_all, 99)

def process_and_save_plot(t):
    fig, ax, cax, _ = plot_lepe_pitchangle_polar(
        da_LEPe=da_LEPi_HE,
        time=t,
        ylabel=ylabel_lepi_HE,
        vmin=vmin_global,
        vmax=vmax_global,
        mass=1.6726219e-27 * 4.,
        VDF_convert=True,
    )
    if fig is None:
        return None

    ts = pd.Timestamp(t)
    base = ts.strftime('%Y-%m-%dT%H%M%S')
    png_path = os.path.join(out_dir_png, base + '.png')
    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

    try:
        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
    finally:
        plt.close(fig)

    return

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']

t_min, t_max    = pd.to_datetime(time_range)
time_grid = da_LEPi_HE.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
    results_list = Parallel(n_jobs=os.cpu_count(), backend="loky", verbose=0)(
        delayed(process_and_save_plot)(t) for t in time_grid
    )
print('Finished saving all plots!')

# $\mathrm{O}^{+}$

In [ ]:
energy_list_O = psp.get_data('erg_lepi_l2_3dflux_FODU', xarray=True).v1 * 1E3

energy_list_O = np.unique(np.sort(energy_list_O))

print(energy_list_O)

In [ ]:
for i, energy in enumerate(energy_list_O):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FODU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_O, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FODU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FODU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FODU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FODU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_O = xr.open_dataset(LEPi_FODU_flux_Path)

print(da_LEPi_O)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

ylabel_lepi_O = r"$\mathrm{O}^{+}$ Energy [eV]"

out_dir = f'/mnt/j/KAW_observation/LEP-i_O_pitch_angle_each_time/20220901/21-24_energy_pa_VDF'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

flux_all    = np.where(da_LEPi_O.FEDU_flux.values > 0, da_LEPi_O.FEDU_flux.values, np.nan)
VDF_all     = flux_all * 5E3 / da_LEPi_O["energy"].values[:, None] * (1.6726219e-27 * 16. / 1.60218E-19)**2.  # [s3 m-6]
vmin_global = np.nanpercentile(VDF_all, 50)
vmax_global = np.nanpercentile(VDF_all, 99)

def process_and_save_plot(t):
    fig, ax, cax, _ = plot_lepe_pitchangle_polar(
        da_LEPe=da_LEPi_O,
        time=t,
        ylabel=ylabel_lepi_O,
        vmin=vmin_global,
        vmax=vmax_global,
        mass=1.6726219e-27 * 16.,
        VDF_convert=True,
    )
    if fig is None:
        return None

    ts = pd.Timestamp(t)
    base = ts.strftime('%Y-%m-%dT%H%M%S')
    png_path = os.path.join(out_dir_png, base + '.png')
    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

    try:
        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
    finally:
        plt.close(fig)

    return

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']

t_min, t_max    = pd.to_datetime(time_range)
time_grid = da_LEPi_O.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
        delayed(process_and_save_plot)(t) for t in time_grid
    )
print('Finished saving all plots!')